# IP Subnetting — interactive tool

Solves every exercise type from the lab
**[cybercloud.upb.ro/docs/network/ip_subnetting](https://cybercloud.upb.ro/docs/network/ip_subnetting)**
from a given input.

**How to use:** run the *Engine* cell once, then jump to whatever you need:
network/broadcast for an IP, binary conversion, mask for N hosts, fixed-size
subnetting, or VLSM. Each section has a single input you edit and re-run.

| Calculation | Formula (from the lab) |
|---|---|
| Network address | `IP AND mask` (host bits → 0) |
| Broadcast address | `IP OR (NOT mask)` (host bits → 1) |
| Usable hosts | `2^(host bits) − 2` |
| Subnets created | `2^(borrowed bits)` |

> Subnet mask = 4 octets of bits that **start with 1 and are all consecutive**
> (e.g. `255.255.240.0` = `/20`). Host bits are the trailing `0`s.


## ⚙️ Engine — run this first
Pure-Python implementation (no external packages). Every value is computed with the bitwise AND/OR from the lab, not a library.

In [1]:
# ---------- low-level helpers ----------
def ip_to_int(ip):
    parts = [int(o) for o in ip.split(".")]
    assert len(parts) == 4 and all(0 <= o <= 255 for o in parts), f"bad IP {ip}"
    return (parts[0] << 24) | (parts[1] << 16) | (parts[2] << 8) | parts[3]

def int_to_ip(n):
    return ".".join(str((n >> s) & 0xFF) for s in (24, 16, 8, 0))

def ip_to_bin(ip):
    "Dotted IP -> dotted 8-bit binary, e.g. 127 -> 01111111"
    return ".".join(f"{int(o):08b}" for o in ip.split("."))

def mask_from_prefix(p):
    assert 0 <= p <= 32, f"bad prefix /{p}"
    return (0xFFFFFFFF << (32 - p)) & 0xFFFFFFFF if p else 0

def prefix_from_mask(mask_int):
    bits = f"{mask_int:032b}"
    assert "01" not in bits, f"non-contiguous mask {int_to_ip(mask_int)}"
    return bits.count("1")

def parse(ip_cidr):
    """Accepts '172.16.200.100/20', '172.16.200.100 255.255.240.0', or bare IP."""
    ip_cidr = ip_cidr.strip()
    if "/" in ip_cidr:
        ip, p = ip_cidr.split("/");           return ip_to_int(ip), int(p)
    if " " in ip_cidr:
        ip, m = ip_cidr.split();              return ip_to_int(ip), prefix_from_mask(ip_to_int(m))
    return ip_to_int(ip_cidr), 32

# ---------- main analyzer ----------
def subnet_report(ip_cidr):
    ip, p   = parse(ip_cidr)
    mask    = mask_from_prefix(p)
    wild    = (~mask) & 0xFFFFFFFF
    network = ip & mask            # IP AND mask
    bcast   = network | wild       # network OR inverted mask
    host_bits = 32 - p
    total   = 2 ** host_bits
    if   p == 32: first = last = ip;                usable = 1
    elif p == 31: first, last = network, bcast;     usable = 2
    else:         first, last = network + 1, bcast - 1; usable = total - 2
    return {
        "ip": int_to_ip(ip), "prefix": p, "mask": int_to_ip(mask),
        "wildcard": int_to_ip(wild), "network": int_to_ip(network),
        "broadcast": int_to_ip(bcast), "first_host": int_to_ip(first),
        "last_host": int_to_ip(last), "total_addresses": total,
        "usable_hosts": usable, "host_bits": host_bits,
    }

def print_report(ip_cidr):
    r = subnet_report(ip_cidr)
    print(f"Input            {ip_cidr}")
    print(f"Address          {r['ip']}")
    print(f"Netmask          {r['mask']}  (/{r['prefix']})")
    print(f"Wildcard         {r['wildcard']}")
    print(f"Network          {r['network']}/{r['prefix']}")
    print(f"Broadcast        {r['broadcast']}")
    print(f"Host range       {r['first_host']}  ...  {r['last_host']}")
    print(f"Usable hosts     {r['usable_hosts']}   (2^{r['host_bits']} - 2)")
    print()
    print(f"  IP    {ip_to_bin(r['ip'])}")
    print(f"  mask  {ip_to_bin(r['mask'])}")
    print(f"  net   {ip_to_bin(r['network'])}  (AND)")
    print(f"  bcast {ip_to_bin(r['broadcast'])}  (OR ~mask)")
    return r

# ---------- planners ----------
def mask_for_hosts(n):
    "Smallest prefix whose usable hosts >= n"
    bits = 1
    while (2 ** bits - 2) < n: bits += 1
    return 32 - bits

def mask_for_subnets(n):
    "Borrowed bits needed to create n subnets"
    bits = 0
    while (2 ** bits) < n: bits += 1
    return bits

def fixed_subnets(base_cidr, new_prefix):
    "Split base network into equal /new_prefix subnets"
    ip, p   = parse(base_cidr)
    network = ip & mask_from_prefix(p)
    step    = 2 ** (32 - new_prefix)
    count   = 2 ** (new_prefix - p)
    return [f"{int_to_ip(network + i*step)}/{new_prefix}" for i in range(count)]

def vlsm(base_cidr, requirements):
    "requirements = [(name, hosts), ...] -> allocations, largest-first"
    ip, p  = parse(base_cidr)
    cursor = ip & mask_from_prefix(p)
    out = []
    for name, hosts in sorted(requirements, key=lambda r: -r[1]):
        np   = mask_for_hosts(hosts)
        rep  = subnet_report(f"{int_to_ip(cursor)}/{np}")
        out.append((name, hosts, rep))
        cursor += 2 ** (32 - np)
    return out

print("Engine loaded. ip_to_bin / print_report / mask_for_hosts / fixed_subnets / vlsm ready.")


Engine loaded. ip_to_bin / print_report / mask_for_hosts / fixed_subnets / vlsm ready.


## 1. Network / broadcast / hosts from an IP
Edit `INPUT` — accepts `IP/prefix`, `IP mask`, or a bare IP.
*(Lab example `172.16.200.100/20` → network `172.16.192.0`, broadcast `172.16.207.255`.)*

In [2]:
INPUT = "172.16.200.100/20"

print_report(INPUT);


Input            172.16.200.100/20
Address          172.16.200.100
Netmask          255.255.240.0  (/20)
Wildcard         0.0.15.255
Network          172.16.192.0/20
Broadcast        172.16.207.255
Host range       172.16.192.1  ...  172.16.207.254
Usable hosts     4094   (2^12 - 2)

  IP    10101100.00010000.11001000.01100100
  mask  11111111.11111111.11110000.00000000
  net   10101100.00010000.11000000.00000000  (AND)
  bcast 10101100.00010000.11001111.11111111  (OR ~mask)


## 2. Binary conversion
Lab exercise: convert numbers like 127, 80, 72, 254 to 8-bit binary.

In [3]:
NUMBERS = [127, 80, 72, 254]

for n in NUMBERS:
    print(f"{n:>3} = {n:08b}")

# whole IP:
print("\n127.80.72.254 =", ip_to_bin("127.80.72.254"))


127 = 01111111
 80 = 01010000
 72 = 01001000
254 = 11111110

127.80.72.254 = 01111111.01010000.01001000.11111110


## 3. Which mask do I need?
- `mask_for_hosts(n)` → smallest subnet that fits **n hosts**
- `mask_for_subnets(n)` → bits to borrow to create **n subnets**

In [4]:
HOSTS_NEEDED   = 50
SUBNETS_NEEDED = 5

p = mask_for_hosts(HOSTS_NEEDED)
print(f"{HOSTS_NEEDED} hosts -> /{p} = {int_to_ip(mask_from_prefix(p))} "
      f"({2**(32-p)-2} usable)")

b = mask_for_subnets(SUBNETS_NEEDED)
print(f"{SUBNETS_NEEDED} subnets -> borrow {b} bits (2^{b} = {2**b} subnets)")


50 hosts -> /26 = 255.255.255.192 (62 usable)
5 subnets -> borrow 3 bits (2^3 = 8 subnets)


## 4. Fixed-size subnetting
Split one network into equal subnets (e.g. a `/24` into `/26`s).

In [5]:
BASE       = "192.168.1.0/24"
NEW_PREFIX = 26

for net in fixed_subnets(BASE, NEW_PREFIX):
    r = subnet_report(net)
    print(f"{net:<18} hosts {r['first_host']}-{r['last_host']:<15} bcast {r['broadcast']}")


192.168.1.0/26     hosts 192.168.1.1-192.168.1.62    bcast 192.168.1.63
192.168.1.64/26    hosts 192.168.1.65-192.168.1.126   bcast 192.168.1.127
192.168.1.128/26   hosts 192.168.1.129-192.168.1.190   bcast 192.168.1.191
192.168.1.192/26   hosts 192.168.1.193-192.168.1.254   bcast 192.168.1.255


## 5. VLSM — variable length subnetting
Sort requirements largest-first, allocate from the top of the block.
*(Lab example `17.18.19.0/24` with needs 100/50/50 → `/25, /26, /26`.)*

In [6]:
BASE = "17.18.19.0/24"
REQUIREMENTS = [("A", 100), ("B", 50), ("C", 50)]   # (name, hosts)

for name, need, r in vlsm(BASE, REQUIREMENTS):
    print(f"{name}: need {need:>4}  ->  {r['network']}/{r['prefix']:<3} "
          f"mask {r['mask']:<15} range {r['first_host']}-{r['last_host']:<15} "
          f"bcast {r['broadcast']}  ({r['usable_hosts']} usable)")


A: need  100  ->  17.18.19.0/25  mask 255.255.255.128 range 17.18.19.1-17.18.19.126    bcast 17.18.19.127  (126 usable)
B: need   50  ->  17.18.19.128/26  mask 255.255.255.192 range 17.18.19.129-17.18.19.190    bcast 17.18.19.191  (62 usable)
C: need   50  ->  17.18.19.192/26  mask 255.255.255.192 range 17.18.19.193-17.18.19.254    bcast 17.18.19.255  (62 usable)


---
### Quick reference — prefix ↔ hosts

| /n | mask | hosts | | /n | mask | hosts |
|----|------|-------|-|----|------|-------|
| /24 | 255.255.255.0 | 254 | | /28 | 255.255.255.240 | 14 |
| /25 | 255.255.255.128 | 126 | | /29 | 255.255.255.248 | 6 |
| /26 | 255.255.255.192 | 62 | | /30 | 255.255.255.252 | 2 |
| /27 | 255.255.255.224 | 30 | | /32 | 255.255.255.255 | 1 |
